# TP 2 — Comprendre `model.py` en profondeur — ÉNONCÉ

## Objectif général

Ce TP est le cœur du parcours.

Dans ce notebook, tu apprendras à comprendre le fond du fonctionnement d'un mini-LLM GPT à partir de `model.py` de nanoGPT.  
Tu ne vas pas seulement exécuter du code : tu vas suivre les tenseurs, les formes, les opérations mathématiques et leur rôle dans un LLM.

À la fin, tu dois être capable d'expliquer le trajet :

`idx → token embeddings → position embeddings → Block → attention causale → MLP → logits → loss`

Règle de travail : pour chaque ligne importante de `model.py`, demande-toi :
1. quelle est la forme du tenseur ;
2. quelle opération mathématique est effectuée ;
3. pourquoi cette opération est nécessaire ;
4. ce qui change entre entraînement et inférence.


## Sources principales du parcours

Ces notebooks se basent principalement sur nanoGPT de Karpathy et sur la documentation officielle PyTorch/Jupyter.

- nanoGPT — README et quick start : https://github.com/karpathy/nanoGPT#quick-start  
  À lire pour comprendre le flux officiel : préparer Shakespeare, entraîner avec `train.py`, puis générer avec `sample.py`.
- nanoGPT — `model.py` : https://github.com/karpathy/nanoGPT/blob/master/model.py  
  À lire comme colonne vertébrale : `LayerNorm`, `CausalSelfAttention`, `MLP`, `Block`, `GPTConfig`, `GPT.forward`, `GPT.generate`.
- nanoGPT — `train.py` : https://github.com/karpathy/nanoGPT/blob/master/train.py  
  À lire pour la boucle d'entraînement : batch, forward, loss, backward, optimizer, checkpoint.
- nanoGPT — `sample.py` : https://github.com/karpathy/nanoGPT/blob/master/sample.py  
  À lire pour l'inférence : chargement du checkpoint, encodage du prompt, `model.generate`.
- Config Shakespeare caractère : https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py  
  À lire pour comprendre les hyperparamètres pédagogiques : `block_size`, `n_layer`, `n_head`, `n_embd`, `dropout`, `max_iters`.
- PyTorch — Tensors : https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html  
  À lire pour comprendre pourquoi les données, poids et activations sont des tenseurs.
- PyTorch — `torch.nn.Embedding` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html  
  À lire pour comprendre la table qui transforme des indices de tokens en vecteurs.
- PyTorch — `torch.nn.LayerNorm` : https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html  
  À lire pour comprendre la normalisation utilisée dans chaque bloc.
- PyTorch — `torch.nn.GELU` : https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html  
  À lire pour comprendre l'activation du MLP.
- PyTorch — `torch.nn.CrossEntropyLoss` : https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html  
  À lire pour comprendre la loss de prédiction du prochain token.
- PyTorch — `torch.nn.functional.scaled_dot_product_attention` : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html  
  À lire pour comprendre la version optimisée de l'attention Q/K/V.
- PyTorch — CUDA semantics : https://docs.pytorch.org/docs/stable/notes/cuda.html  
  À lire pour comprendre le placement CPU/GPU des tenseurs et modèles.
- Jupyter — Markdown cells : https://jupyter-notebook.readthedocs.io/en/stable/examples/Notebook/Working%20With%20Markdown%20Cells.html  
  À lire pour savoir structurer les réponses dans les cellules Markdown.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print("PyTorch:", torch.__version__)


## Partie 1 — Les dimensions fondamentales

Dans cette partie, tu apprendras à lire les formes des tenseurs.  
C'est indispensable : si tu ne comprends pas les shapes, tu ne peux pas comprendre l'attention.


### Question 2.1 — Comprendre les dimensions `(B, T, C)`

**Objectif de la partie :**  
Comprendre le vocabulaire des dimensions utilisées dans nanoGPT.

**Question :**  
Dans un tenseur de forme `(B, T, C)`, explique ce que représentent `B`, `T` et `C`. Donne un exemple concret avec `B=2`, `T=4`, `C=8`.

**Prérequis :**
- Notion de tenseur
- Notion de batch
- Notion de séquence de tokens

**Sources exactes à lire avant de répondre :**
1. PyTorch tensors tutorial : https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html — lire l'introduction.
2. nanoGPT `model.py` : https://github.com/karpathy/nanoGPT/blob/master/model.py — chercher `b, t = idx.size()` dans `GPT.forward`.

**Ce que tu dois chercher dans ces sources :**
- Ce qu'est un tenseur PyTorch.
- Pourquoi `idx` a deux dimensions au départ.
- Pourquoi les activations internes ont souvent trois dimensions.

**Réponse :**


In [ ]:
# À compléter
B, T, C = ...
x = torch.randn(B, T, C)
print(x.shape)


**Indice :**  
Dans nanoGPT, `C` correspond généralement à `config.n_embd`.


### Question 2.2 — Comprendre `idx` et `targets`

**Objectif de la partie :**  
Comprendre ce que le modèle reçoit pendant l'entraînement.

**Question :**  
Dans `GPT.forward(self, idx, targets=None)`, explique la différence entre `idx` et `targets`. Pourquoi ont-ils souvent la même forme `(B, T)` ?

**Prérequis :**
- Comprendre la prédiction du prochain token
- Comprendre la notion de séquence d'entraînement

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher la signature `def forward(self, idx, targets=None)`.
2. nanoGPT `train.py` : chercher `x = ...` et `y = ...` dans `get_batch`.
3. PyTorch CrossEntropyLoss : https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html — lire le début.

**Ce que tu dois chercher dans ces sources :**
- Comment `x` et `y` sont construits dans `train.py`.
- Pourquoi `y` est décalé d'un token par rapport à `x`.
- Pourquoi la loss a besoin d'une cible.

**Réponse :**


In [ ]:
# À compléter : crée un exemple jouet
sequence = torch.tensor([0, 1, 2, 3, 4, 5])
# idx = ...
# targets = ...
# print(...)


**Indice :**  
`idx` est ce qu'on donne au modèle ; `targets` est ce qu'on veut qu'il prédise.


## Partie 2 — `GPTConfig` et les hyperparamètres

Dans cette partie, tu apprendras à relier les nombres de la configuration à la taille du modèle et aux formes des tenseurs.


### Question 2.3 — Lire `GPTConfig`

**Objectif de la partie :**  
Comprendre les hyperparamètres minimaux qui définissent un GPT.

**Question :**  
Dans `GPTConfig`, explique le rôle de `block_size`, `vocab_size`, `n_layer`, `n_head`, `n_embd`, `dropout` et `bias`.

**Prérequis :**
- Notion d'hyperparamètre
- Notion de vocabulaire
- Notion de longueur de contexte

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `@dataclass` puis `class GPTConfig`.
2. Config Shakespeare : https://github.com/karpathy/nanoGPT/blob/master/config/train_shakespeare_char.py — lire les lignes où `block_size`, `n_layer`, `n_head`, `n_embd` sont définis.

**Ce que tu dois chercher dans ces sources :**
- Quelle valeur contrôle la longueur maximale du contexte.
- Quelle valeur contrôle la dimension des vecteurs.
- Quelle valeur contrôle le nombre de couches Transformer.

**Réponse :**


<!-- Écris ta réponse ici. -->


**Indice :**  
`n_embd` doit être divisible par `n_head`, car chaque tête reçoit une partie de la dimension.


### Question 2.4 — Calculer `head_size`

**Objectif de la partie :**  
Comprendre comment la dimension d'embedding est répartie entre les têtes d'attention.

**Question :**  
Si `n_embd = 8` et `n_head = 2`, quelle est la taille d'une tête ? Pourquoi nanoGPT impose implicitement que `n_embd % n_head == 0` ?

**Prérequis :**
- Division entière
- Notion de multi-head attention

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `assert config.n_embd % config.n_head == 0`.
2. PyTorch SDPA : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html — repérer les dimensions `query`, `key`, `value`.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi on découpe `C` en plusieurs sous-espaces.
- Pourquoi chaque tête travaille sur `C / n_head` dimensions.

**Réponse :**


In [ ]:
# À compléter
n_embd = ...
n_head = ...
head_size = ...
print(head_size)


**Indice :**  
Chaque tête ne voit qu'une tranche de la dimension totale.


## Partie 3 — Embeddings : passer des tokens aux vecteurs

Dans cette partie, tu apprendras pourquoi un LLM ne travaille pas directement sur du texte, mais sur des vecteurs appris.


### Question 2.5 — Comprendre les token embeddings

**Objectif de la partie :**  
Comprendre comment un identifiant entier devient un vecteur.

**Question :**  
Dans `GPT.forward`, explique ce que fait `tok_emb = self.transformer.wte(idx)`. Quelle est la forme de sortie si `idx` a la forme `(B, T)` et `n_embd=C` ?

**Prérequis :**
- Savoir ce qu'est un vocabulaire
- Comprendre qu'un token est représenté par un entier

**Sources exactes à lire avant de répondre :**
1. PyTorch `nn.Embedding` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html — lire la première description.
2. nanoGPT `model.py` : chercher `wte = nn.Embedding(config.vocab_size, config.n_embd)` puis `tok_emb = self.transformer.wte(idx)`.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi `nn.Embedding` est une table de lookup.
- Pourquoi la sortie gagne une dimension `C`.
- Pourquoi un indice seul ne contient pas le sens du token.

**Réponse :**


In [ ]:
# À compléter
B, T, C, vocab_size = ...
idx = torch.randint(0, vocab_size, (B, T))
wte = nn.Embedding(vocab_size, C)
tok_emb = ...
print("idx:", idx.shape)
print("tok_emb:", tok_emb.shape)


**Indice :**  
Imagine une table avec `vocab_size` lignes et `n_embd` colonnes.


### Question 2.6 — Comprendre les position embeddings

**Objectif de la partie :**  
Comprendre pourquoi le modèle doit connaître la position des tokens.

**Question :**  
Explique ce que fait `pos_emb = self.transformer.wpe(pos)`. Pourquoi un Transformer a besoin d'information de position ?

**Prérequis :**
- Comprendre qu'une séquence a un ordre
- Comprendre qu'une attention brute compare des tokens sans notion explicite de position

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `wpe = nn.Embedding(config.block_size, config.n_embd)` et `pos = torch.arange(0, t, dtype=torch.long, device=device)`.
2. The Illustrated Transformer : https://jalammar.github.io/illustrated-transformer/ — chercher la section sur positional encoding.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi la position 0 n'a pas le même rôle que la position 10.
- Pourquoi `pos` va de `0` à `T-1`.
- Pourquoi `pos_emb` a la forme `(T, C)`.

**Réponse :**


In [ ]:
# À compléter
T, C, block_size = ...
pos = torch.arange(0, T)
wpe = nn.Embedding(block_size, C)
pos_emb = ...
print("pos:", pos)
print("pos_emb:", pos_emb.shape)


**Indice :**  
`tok_emb` dit quel token est présent ; `pos_emb` dit où il est dans la séquence.


### Question 2.7 — Additionner token et position embeddings

**Objectif de la partie :**  
Comprendre la première représentation interne `x` du modèle.

**Question :**  
Dans nanoGPT, on calcule `x = self.transformer.drop(tok_emb + pos_emb)`. Explique pourquoi on peut additionner `tok_emb` de forme `(B,T,C)` et `pos_emb` de forme `(T,C)`.

**Prérequis :**
- Broadcasting PyTorch
- Embeddings
- Formes de tenseurs

**Sources exactes à lire avant de répondre :**
1. PyTorch tensors tutorial : https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html — revoir les opérations sur tenseurs.
2. nanoGPT `model.py` : chercher `tok_emb + pos_emb` dans `GPT.forward`.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi PyTorch peut diffuser `pos_emb` sur la dimension batch.
- Ce que contient `x` après l'addition.
- Pourquoi cette addition est le point d'entrée du Transformer.

**Réponse :**


In [ ]:
# À compléter
B, T, C = ...
tok_emb = torch.randn(B, T, C)
pos_emb = torch.randn(T, C)
x = ...
print(x.shape)


**Indice :**  
Le même embedding de position est ajouté à tous les exemples du batch.


## Partie 4 — Entrer dans un bloc Transformer

Dans cette partie, tu apprendras la structure d'un bloc GPT : LayerNorm, attention, résidu, MLP, résidu.


### Question 2.8 — Lire la structure de `Block`

**Objectif de la partie :**  
Comprendre l'ordre des opérations dans un bloc Transformer de type GPT.

**Question :**  
Dans la classe `Block`, explique le rôle des deux lignes principales du `forward` : `x = x + self.attn(self.ln_1(x))` et `x = x + self.mlp(self.ln_2(x))`.

**Prérequis :**
- Notion de fonction résiduelle
- LayerNorm
- Attention
- MLP

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `class Block`.
2. PyTorch `LayerNorm` : https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html — lire la description.
3. The Illustrated Transformer : https://jalammar.github.io/illustrated-transformer/ — chercher les blocs encodeur/décodeur et les connexions résiduelles.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi `x + ...` conserve une voie directe.
- Pourquoi nanoGPT normalise avant attention et avant MLP.
- Pourquoi le bloc ne change généralement pas la forme `(B,T,C)`.

**Réponse :**


<!-- Écris ta réponse ici. -->


**Indice :**  
Un bloc transforme le contenu de `x`, mais garde la même forme pour pouvoir empiler plusieurs blocs.


### Question 2.9 — Comprendre LayerNorm

**Objectif de la partie :**  
Comprendre pourquoi on normalise les activations dans chaque bloc.

**Question :**  
Explique ce que fait une LayerNorm sur un tenseur `(B,T,C)`. Sur quelle dimension normalise-t-elle dans nanoGPT ?

**Prérequis :**
- Moyenne
- Variance
- Stabilité d'entraînement

**Sources exactes à lire avant de répondre :**
1. PyTorch `LayerNorm` : https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html — lire la définition et l'argument `normalized_shape`.
2. nanoGPT `model.py` : chercher `LayerNorm(config.n_embd, bias=config.bias)`.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi `normalized_shape=config.n_embd` signifie normaliser sur la dernière dimension.
- Pourquoi cela stabilise les valeurs qui entrent dans attention et MLP.

**Réponse :**


In [ ]:
# À compléter
B, T, C = ...
x = torch.randn(B, T, C)
ln = nn.LayerNorm(C)
y = ...
print(x.shape, y.shape)


**Indice :**  
Pour chaque token, on normalise son vecteur de taille `C`.


## Partie 5 — Attention causale pas à pas

Dans cette partie, tu apprendras le cœur du modèle.  
On va volontairement ralentir : Q, K, V, reshape, score, masque, softmax, combinaison des valeurs.


### Question 2.10 — Projeter `x` en Q, K, V

**Objectif de la partie :**  
Comprendre pourquoi l'attention crée trois vues apprises de `x`.

**Question :**  
Dans `CausalSelfAttention`, explique le rôle de `self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, ...)`. Pourquoi produire `3*C` dimensions ?

**Prérequis :**
- Projection linéaire
- Embedding
- Q/K/V

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `self.c_attn` puis `q, k, v = self.c_attn(x).split(...)`.
2. PyTorch `nn.Linear` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html — lire la description.
3. PyTorch SDPA : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html — repérer `query`, `key`, `value`.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi une seule Linear peut calculer Q, K et V ensemble.
- Pourquoi on découpe ensuite le résultat.
- Quel est le rôle intuitif de Q, K et V.

**Réponse :**


In [ ]:
# À compléter
B, T, C = ...
x = torch.randn(B, T, C)
c_attn = nn.Linear(C, 3*C)
qkv = ...
print(qkv.shape)


**Indice :**  
Q sert à demander, K sert à être comparé, V contient l'information à récupérer.


### Question 2.11 — Découper Q, K, V

**Objectif de la partie :**  
Comprendre la ligne `.split(self.n_embd, dim=2)`.

**Question :**  
À partir d'un tenseur `qkv` de forme `(B,T,3C)`, découpe-le en `q`, `k`, `v` de forme `(B,T,C)`.

**Prérequis :**
- Indexation/split de tenseurs
- Formes de tenseurs

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `.split(self.n_embd, dim=2)`.
2. PyTorch `torch.split` : https://docs.pytorch.org/docs/stable/generated/torch.split.html — lire la description.

**Ce que tu dois chercher dans ces sources :**
- Sur quelle dimension se fait le découpage.
- Pourquoi chaque morceau a taille `C`.
- Pourquoi les trois tenseurs gardent les mêmes `B` et `T`.

**Réponse :**


In [ ]:
# À compléter
B, T, C = ...
qkv = torch.randn(B, T, 3*C)
q, k, v = ...
print(q.shape, k.shape, v.shape)


**Indice :**  
La dimension `2` est la dimension des canaux/features.


### Question 2.12 — Préparer le multi-head attention

**Objectif de la partie :**  
Comprendre pourquoi on reshape et transpose Q, K, V.

**Question :**  
nanoGPT transforme `q` de `(B,T,C)` vers `(B,n_head,T,head_size)`. Reproduis cette transformation et explique pourquoi elle est nécessaire.

**Prérequis :**
- `view` / `reshape`
- `transpose`
- Multi-head attention

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `view(B, T, self.n_head, C // self.n_head).transpose(1, 2)`.
2. PyTorch `Tensor.view` : https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html
3. PyTorch `torch.transpose` : https://docs.pytorch.org/docs/stable/generated/torch.transpose.html

**Ce que tu dois chercher dans ces sources :**
- Pourquoi on sépare `C` en `n_head * head_size`.
- Pourquoi on veut la forme `(B,n_head,T,head_size)` pour l'attention.
- Pourquoi l'ordre des dimensions compte pour les multiplications.

**Réponse :**


In [ ]:
# À compléter
B, T, C, n_head = ...
head_size = ...
q = torch.randn(B, T, C)
q_heads = ...
print(q_heads.shape)


**Indice :**  
Chaque tête doit calculer sa propre matrice d'attention sur la séquence.


### Question 2.13 — Calculer les scores d'attention

**Objectif de la partie :**  
Comprendre le produit `q @ k.transpose(-2, -1)`.

**Question :**  
Si `q` et `k` ont la forme `(B,n_head,T,head_size)`, quelle est la forme de `scores = q @ k.transpose(-2, -1)` ? Que représente chaque valeur `scores[b,h,i,j]` ?

**Prérequis :**
- Produit matriciel
- Transpose
- Attention

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `att = (q @ k.transpose(-2, -1)) * ...`.
2. The Annotated Transformer : https://nlp.seas.harvard.edu/annotated-transformer/ — chercher `query`, `key`, `value` et `attention`.
3. PyTorch matmul : https://docs.pytorch.org/docs/stable/generated/torch.matmul.html

**Ce que tu dois chercher dans ces sources :**
- Pourquoi on obtient une matrice `(T,T)` par tête.
- Pourquoi chaque position compare sa query aux keys de toutes les positions.
- Pourquoi on divise par `sqrt(head_size)`.

**Réponse :**


In [ ]:
# À compléter
B, n_head, T, head_size = ...
q = torch.randn(B, n_head, T, head_size)
k = torch.randn(B, n_head, T, head_size)
scores = ...
print(scores.shape)


**Indice :**  
La dimension `head_size` disparaît dans le produit, car elle est sommée.


### Question 2.14 — Appliquer le masque causal

**Objectif de la partie :**  
Comprendre pourquoi GPT ne doit pas regarder le futur pendant l'entraînement.

**Question :**  
Construis un masque causal pour `T=4`. Explique pourquoi les positions futures doivent être masquées dans un modèle autoregressif.

**Prérequis :**
- Autoregression
- Matrice triangulaire
- Softmax

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `bias` et `masked_fill(..., float('-inf'))`.
2. The Annotated Transformer : https://nlp.seas.harvard.edu/annotated-transformer/ — chercher `subsequent_mask`.
3. The Illustrated Transformer : https://jalammar.github.io/illustrated-transformer/ — chercher l'explication du décodeur et du masquage.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi le token à la position 2 peut voir 0,1,2 mais pas 3.
- Pourquoi on met `-inf` avant softmax.
- Pourquoi cela simule la génération de gauche à droite.

**Réponse :**


In [ ]:
# À compléter
T = ...
mask = ...
print(mask)


**Indice :**  
Après softmax, une valeur `-inf` devient une probabilité 0.


### Question 2.15 — Transformer les scores en probabilités

**Objectif de la partie :**  
Comprendre le rôle du softmax dans l'attention.

**Question :**  
Applique un softmax sur la dernière dimension des scores masqués. Pourquoi la somme sur chaque ligne vaut-elle 1 ?

**Prérequis :**
- Exponentielle
- Probabilité
- Softmax

**Sources exactes à lire avant de répondre :**
1. PyTorch `softmax` : https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html
2. nanoGPT `model.py` : chercher `F.softmax(att, dim=-1)`.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi `dim=-1` correspond aux positions que chaque token peut regarder.
- Pourquoi l'attention devient une distribution de poids.
- Pourquoi les positions futures ont une probabilité 0.

**Réponse :**


In [ ]:
# À compléter
scores = torch.randn(1, 1, 4, 4)
# masked_scores = ...
# att = ...
# print(att)
# print(att.sum(dim=-1))


**Indice :**  
Le softmax transforme des scores quelconques en poids positifs qui somment à 1.


### Question 2.16 — Calculer `att @ v`

**Objectif de la partie :**  
Comprendre comment les poids d'attention récupèrent l'information.

**Question :**  
Si `att` a la forme `(B,n_head,T,T)` et `v` la forme `(B,n_head,T,head_size)`, quelle est la forme de `y = att @ v` ? Explique ce que représente `y[b,h,i]`.

**Prérequis :**
- Produit matriciel
- Combinaison pondérée
- Attention

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `y = att @ v`.
2. The Annotated Transformer : https://nlp.seas.harvard.edu/annotated-transformer/ — chercher la formule `softmax(QK^T/sqrt(d_k))V`.

**Ce que tu dois chercher dans ces sources :**
- Pourquoi la dimension des clés/positions `T` est consommée.
- Pourquoi la sortie contient un vecteur par position.
- Pourquoi chaque vecteur est un mélange des values autorisées.

**Réponse :**


In [ ]:
# À compléter
B, n_head, T, head_size = ...
att = torch.randn(B, n_head, T, T).softmax(dim=-1)
v = torch.randn(B, n_head, T, head_size)
y = ...
print(y.shape)


**Indice :**  
`att[i, j]` dit combien la position `i` récupère d'information depuis la position `j`.


### Question 2.17 — Recombiner les têtes

**Objectif de la partie :**  
Comprendre pourquoi on revient de `(B,n_head,T,head_size)` à `(B,T,C)`.

**Question :**  
Reproduis la transformation `y.transpose(1, 2).contiguous().view(B, T, C)`. Pourquoi faut-il recombiner les têtes ?

**Prérequis :**
- Transpose
- Mémoire contiguë
- Multi-head attention

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `y = y.transpose(1, 2).contiguous().view(B, T, C)`.
2. PyTorch `contiguous` : https://docs.pytorch.org/docs/stable/generated/torch.Tensor.contiguous.html

**Ce que tu dois chercher dans ces sources :**
- Pourquoi on veut revenir à la forme standard `(B,T,C)`.
- Pourquoi les couches suivantes attendent cette forme.
- Pourquoi `.contiguous()` apparaît après un transpose.

**Réponse :**


In [ ]:
# À compléter
B, T, C, n_head = ...
head_size = ...
y = torch.randn(B, n_head, T, head_size)
y_recombined = ...
print(y_recombined.shape)


**Indice :**  
Les têtes travaillent séparément, mais le reste du modèle attend un seul vecteur par token.


## Partie 6 — MLP, logits et loss

Dans cette partie, tu apprendras ce qui se passe après l'attention : transformation non linéaire, projection vers le vocabulaire, puis calcul de la loss.


### Question 2.18 — Comprendre le MLP

**Objectif de la partie :**  
Comprendre pourquoi chaque bloc contient aussi un réseau feed-forward.

**Question :**  
Dans nanoGPT, le MLP fait `Linear(C,4C) → GELU → Linear(4C,C) → Dropout`. Explique pourquoi la forme finale redevient `(B,T,C)`.

**Prérequis :**
- Couche linéaire
- Activation
- GELU
- Dropout

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `class MLP`.
2. PyTorch `GELU` : https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html
3. PyTorch `Dropout` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html

**Ce que tu dois chercher dans ces sources :**
- Pourquoi on agrandit temporairement la dimension à `4C`.
- Pourquoi GELU introduit de la non-linéarité.
- Pourquoi la sortie doit revenir à `C`.

**Réponse :**


In [ ]:
# À compléter
B, T, C = ...
mlp = nn.Sequential(
    nn.Linear(C, 4*C),
    nn.GELU(),
    nn.Linear(4*C, C),
)
x = torch.randn(B, T, C)
y = ...
print(y.shape)


**Indice :**  
L'attention mélange l'information entre positions ; le MLP transforme chaque position individuellement.


### Question 2.19 — Comprendre les logits

**Objectif de la partie :**  
Comprendre comment le modèle produit une distribution sur le vocabulaire.

**Question :**  
Dans `GPT.forward`, explique le rôle de `self.lm_head(x)`. Quelle est la forme des logits en entraînement ?

**Prérequis :**
- Vocabulaire
- Projection linéaire
- Logits

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)` et `logits = self.lm_head(x)`.
2. PyTorch `nn.Linear` : https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html

**Ce que tu dois chercher dans ces sources :**
- Pourquoi on projette de `C` vers `vocab_size`.
- Pourquoi un logit n'est pas encore une probabilité.
- Pourquoi il y a un vecteur de logits pour chaque position.

**Réponse :**


In [ ]:
# À compléter
B, T, C, vocab_size = ...
x = torch.randn(B, T, C)
lm_head = nn.Linear(C, vocab_size, bias=False)
logits = ...
print(logits.shape)


**Indice :**  
À chaque position, le modèle donne un score à chaque token possible du vocabulaire.


### Question 2.20 — Comprendre la cross-entropy loss

**Objectif de la partie :**  
Comprendre comment nanoGPT apprend à prédire le prochain token.

**Question :**  
Explique pourquoi nanoGPT fait `logits.view(-1, logits.size(-1))` et `targets.view(-1)` avant `F.cross_entropy`.

**Prérequis :**
- Cross entropy
- Logits
- Targets
- Flatten

**Sources exactes à lire avant de répondre :**
1. nanoGPT `model.py` : chercher `F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)`.
2. PyTorch CrossEntropyLoss : https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html — lire les formes attendues des entrées.
3. PyTorch `view` : https://docs.pytorch.org/docs/stable/generated/torch.Tensor.view.html

**Ce que tu dois chercher dans ces sources :**
- Pourquoi la loss veut une liste de prédictions et une liste de classes cibles.
- Pourquoi `(B,T,V)` devient `(B*T,V)`.
- Pourquoi `(B,T)` devient `(B*T)`.

**Réponse :**


In [ ]:
# À compléter
B, T, vocab_size = ...
logits = torch.randn(B, T, vocab_size)
targets = torch.randint(0, vocab_size, (B, T))
loss = ...
print(loss)


**Indice :**  
La loss compare chaque position indépendamment : prédiction du prochain token à cette position.


## Partie 7 — Synthèse

### Question 2.21 — Raconter le forward pass complet

**Question :**  
Avec tes mots, raconte le forward pass complet de nanoGPT, depuis `idx` jusqu'à `loss`.

**Prérequis :**
- Avoir répondu aux questions précédentes.

**Sources exactes à lire avant de répondre :**
- nanoGPT `model.py` : relire `GPT.forward`.
- Tes propres sorties de `print(shape)` dans ce notebook.

**Réponse :**

<!-- Écris ici ton explication complète. -->

**Indice :**  
Utilise cette trame : tokens → embeddings → positions → blocks → norm finale → lm_head → logits → cross-entropy.
